# Exploratory Analysis: Modeling GenAI Productivity & U.S. Life Expectancy

Use **Managed Service for Apache Spark** and the `lakehouse-public-data.world_bank_wdi` BigLake Iceberg dataset (`7.87M` rows) to model global life expectancy across economic, R&D, and healthcare channels, then run a distributed semiparametric country-cluster bootstrap to estimate the 10-year U.S. longevity impact of generative AI productivity growth (`+0.1%/yr` to `+2.0%/yr` above baseline U.S. GDP growth).

In [ ]:
PROJECT_ID = "YOUR-PROJECT-ID"
REGION = "GOOGLE_CLOUD_REGION"
RUNTIME_PROFILE = "YOUR-RUNTIME-PROFILE"

In [ ]:
# %pip install 'google-cloud-spark-connect[client,interactive]~=2.2' matplotlib
from google.cloud.managed_spark_connect import ManagedSparkSession

spark = (
    ManagedSparkSession.builder
    .projectId(PROJECT_ID)
    .location(REGION)
    .runtimeProfile(RUNTIME_PROFILE)
    .getOrCreate()
)

## 1. Inspect the World Bank WDI Lakehouse Dataset

In [ ]:
%%sparksql
SELECT
  COUNT(*) AS total_observations,
  COUNT(DISTINCT country_code) AS country_and_region_entities,
  COUNT(DISTINCT indicator_code) AS distinct_indicators,
  MIN(year) AS first_year,
  MAX(year) AS last_year
FROM `lakehouse-public-data`.world_bank_wdi.indicators_data

## 2. Engineer Multi-Channel Country-Year Panel Features

To separate static income effects from medium-run growth and innovation channels, we broadcast-join `country_summary` (excluding the 48 regional/income aggregates) and pivot 7 WDI indicators across `1995–2019` (`4,667` country-years):
- **Output & Growth Momentum**: Log PPP GDP per capita (`log_gdp_pcap`) and 5-year rolling GDP-per-capita CAGR (`gdp_5yr_cagr_pct`) via Spark `Window` functions.
- **Innovation & Healthcare Channels**: R&D expenditure (`rd_pct_gdp`) and health expenditure (`health_exp_pct_gdp`) as shares of GDP, plus internet adoption, tertiary education, and urbanization.
- **Causal Forward-Fill**: Biennial UNESCO/OECD survey series (`rd_pct_gdp`, `tertiary_enroll_pct`) are forward-filled within each country (`rowsBetween(unboundedPreceding, 0)`) to avoid listwise deletion bias.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

wdi = spark.read.format("iceberg").load(
    "`lakehouse-public-data`.world_bank_wdi.indicators_data"
)
countries = (
    spark.read.format("iceberg")
    .load("`lakehouse-public-data`.world_bank_wdi.country_summary")
    .filter(F.col("region").isNotNull() & (F.col("region") != ""))
    .select("country_code", "short_name", "region", "income_group")
)

# Select 7 complementary macroeconomic, innovation, and health-system indicators:
# - NY.GDP.PCAP.PP.KD / NY.GDP.PCAP.KD: Real economic output per capita (PPP preferred, constant USD fallback)
# - GB.XPD.RSDV.GD.ZS: Gross domestic R&D expenditure (% of GDP) — proxy for scientific/biomedical innovation intensity
# - SH.XPD.CHEX.GD.ZS: Current health expenditure (% of GDP) — healthcare system resource allocation
# - IT.NET.USER.ZS: Internet penetration (% of population) — digital infrastructure diffusion
# - SE.TER.ENRR / SP.URB.TOTL.IN.ZS: Human capital (tertiary enrollment) and urbanization controls
indicators = {
    "SP.DYN.LE00.IN": "life_expectancy",
    "NY.GDP.PCAP.PP.KD": "gdp_pcap_ppp",
    "NY.GDP.PCAP.KD": "gdp_pcap_const",
    "GB.XPD.RSDV.GD.ZS": "rd_pct_gdp",
    "SH.XPD.CHEX.GD.ZS": "health_exp_pct_gdp",
    "IT.NET.USER.ZS": "internet_users_pct",
    "SE.TER.ENRR": "tertiary_enroll_pct",
    "SP.URB.TOTL.IN.ZS": "urban_pop_pct",
}

panel_raw = (
    wdi.filter(F.col("indicator_code").isin(list(indicators.keys())))
    .join(F.broadcast(countries), on="country_code", how="inner")
    .groupBy("country_code", "short_name", "region", "income_group", "year")
    .pivot("indicator_code", list(indicators.keys()))
    .agg(F.first("value"))
)
for code, col_name in indicators.items():
    panel_raw = panel_raw.withColumnRenamed(code, col_name)

country_window = Window.partitionBy("country_code").orderBy("year")
ffill_window = country_window.rowsBetween(Window.unboundedPreceding, 0)

panel_df = (
    panel_raw
    .withColumn("gdp_pcap", F.coalesce(F.col("gdp_pcap_ppp"), F.col("gdp_pcap_const")))
    .filter(F.col("life_expectancy").isNotNull() & (F.col("gdp_pcap") > 0))
    # Log-transform GDP per capita to linearize the diminishing-returns Preston curve
    .withColumn("log_gdp_pcap", F.log(F.col("gdp_pcap")))
    # Compute 5-year rolling compound annual GDP growth rate (CAGR %) to separate growth momentum from static income levels
    .withColumn("gdp_lag5", F.lag("gdp_pcap", 5).over(country_window))
    .withColumn(
        "gdp_5yr_cagr_pct",
        F.when(
            F.col("gdp_lag5") > 0,
            (F.pow(F.col("gdp_pcap") / F.col("gdp_lag5"), 0.2) - 1.0) * 100.0,
        ),
    )
    # UNESCO/OECD R&D and UNESCO tertiary surveys are reported every 2-3 years in several countries;
    # forward-filling within each country's time series preserves slow-moving structural shares without future leakage.
    .withColumn("rd_pct_gdp", F.last("rd_pct_gdp", ignorenulls=True).over(ffill_window))
    .withColumn("health_exp_pct_gdp", F.last("health_exp_pct_gdp", ignorenulls=True).over(ffill_window))
    .withColumn("tertiary_enroll_pct", F.last("tertiary_enroll_pct", ignorenulls=True).over(ffill_window))
)

# Restrict to 1995-2019 (when UNESCO R&D and WHO health expenditure reporting standardized globally)
# and coalesce post-shuffle partitions down to 8 before caching for low-latency iterative SparkML fits.
train_df = (
    panel_df.filter((F.col("year") >= 1995) & (F.col("year") <= 2019))
    .coalesce(8)
    .cache()
)
print(f"Modern country-year training panel (1995-2019): {train_df.count():,} rows")
train_df.filter((F.col("country_code") == "USA") & (F.col("year") >= 2015)).select(
    "country_code",
    "year",
    F.round("life_expectancy", 2).alias("life_expectancy"),
    F.round("gdp_pcap", 0).alias("gdp_pcap_ppp"),
    F.round("gdp_5yr_cagr_pct", 2).alias("gdp_5yr_cagr_pct"),
    F.round("rd_pct_gdp", 2).alias("rd_pct_gdp"),
    F.round("health_exp_pct_gdp", 2).alias("health_exp_pct_gdp"),
).orderBy("year").show(truncate=False)

In [ ]:
explore_dataframe(
    train_df.filter(F.col("year") == 2019).select(
        "short_name",
        "region",
        "income_group",
        "gdp_pcap",
        "life_expectancy",
        "rd_pct_gdp",
        "health_exp_pct_gdp",
        "internet_users_pct",
    ),
    chart_type="scatter",
)

## 3. Parallel SparkML Cross-Validation Over Spark Connect

We expand the 7 predictors via degree-2 `PolynomialExpansion` (35 linear, quadratic, and pairwise interaction terms) and tune an L1/L2 ElasticNet regression with `CrossValidator(parallelism=2)`. Comparing high-income economies in 2019 highlights a distinct R&D frontier: peers investing `3.17%–3.40%` of GDP in R&D (`AUT`, `DEU`, `JPN`, `CHE`, `SWE`) achieve `81.0–84.4` years of life expectancy versus `78.79` years in the U.S. (`2.83%` R&D).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import Imputer, PolynomialExpansion, StandardScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

feature_cols = [
    "log_gdp_pcap",
    "gdp_5yr_cagr_pct",
    "rd_pct_gdp",
    "health_exp_pct_gdp",
    "internet_users_pct",
    "tertiary_enroll_pct",
    "urban_pop_pct",
]
imputed_cols = [f"{c}_imp" for c in feature_cols]

# Degree-2 PolynomialExpansion generates all 7 linear terms + 28 quadratic/pairwise interaction terms
# (e.g., log_gdp_pcap * rd_pct_gdp, rd_pct_gdp * health_exp_pct_gdp), capturing how R&D and health
# returns depend on baseline economic development. StandardScaler normalizes features for ElasticNet penalties.
feature_pipe = Pipeline(stages=[
    Imputer(inputCols=feature_cols, outputCols=imputed_cols, strategy="median"),
    VectorAssembler(inputCols=imputed_cols, outputCol="raw_features", handleInvalid="keep"),
    PolynomialExpansion(degree=2, inputCol="raw_features", outputCol="interaction_features"),
    StandardScaler(inputCol="interaction_features", outputCol="features", withMean=True, withStd=True),
])
prepared_df = feature_pipe.fit(train_df).transform(train_df).coalesce(8).cache()
_ = prepared_df.count()

# Tune L1/L2 ElasticNet regularization across 3 folds in parallel on Spark executors to prevent collinear overfitting
elastic_net = LinearRegression(featuresCol="features", labelCol="life_expectancy", maxIter=100)
param_grid = (
    ParamGridBuilder()
    .addGrid(elastic_net.regParam, [0.01, 0.05, 0.2])
    .addGrid(elastic_net.elasticNetParam, [0.0, 0.5])
    .build()
)

rmse_eval = RegressionEvaluator(labelCol="life_expectancy", predictionCol="prediction", metricName="rmse")
r2_eval = RegressionEvaluator(labelCol="life_expectancy", predictionCol="prediction", metricName="r2")

cv = CrossValidator(
    estimator=elastic_net,
    estimatorParamMaps=param_grid,
    evaluator=rmse_eval,
    numFolds=3,
    parallelism=2,
    seed=42,
)
cv_model = cv.fit(prepared_df)
cv_preds = cv_model.transform(prepared_df)

print(
    f"Parallel ElasticNet Interaction Model -> Best 3-Fold CV RMSE: {min(cv_model.avgMetrics):.3f} yrs | "
    f"In-sample R2: {r2_eval.evaluate(cv_preds):.3f}"
)

# Inspect high-income peer R&D intensity vs. life expectancy regimes (2.8% to 3.4% of GDP)
cv_preds.filter(
    (F.col("year") == 2019)
    & F.col("country_code").isin("USA", "AUT", "DNK", "DEU", "SWE", "CHE", "JPN")
).select(
    "country_code",
    "short_name",
    F.round("gdp_pcap", 0).alias("gdp_pcap_ppp"),
    F.round("rd_pct_gdp", 2).alias("rd_pct_gdp"),
    F.round("life_expectancy", 2).alias("actual_le"),
    F.round("prediction", 2).alias("predicted_le"),
).orderBy("rd_pct_gdp").show(truncate=False)

## 4. Distributed Semiparametric Country-Cluster Bootstrap (`mapInPandas`, `N=10,000`)

To forecast how a generative AI productivity boom could affect U.S. life expectancy over a 10-year horizon (`HORIZON_YEARS = 10`) with smooth tail percentiles and zero arbitrary scaling constants, we distribute `N = 10,000` **country-cluster bootstrap** replicates (`420,000` scenario evaluations) across 64 Spark partitions (`mapInPandas`). Resampling whole countries with replacement preserves within-country serial correlation.

### Econometric Specification
1. **Explicit Non-AI Baseline (`g_ai = +0.0%`)**: Even without AI, U.S. GDP per capita grows at its empirical 5-year rolling CAGR `g_base = 1.83%/yr` (`$62.6k → $75.1k` over 10 years, reaching `79.23` yrs under balanced growth with constant expenditure shares). Each AI scenario adds `g_ai ∈ [+0.1%, +2.0%]/yr` **on top of** `g_base`, measuring the incremental longevity gain attributable to AI above the `79.23`-yr non-AI baseline.
2. **Stage 1 — Parametric Preston Panel (`Pure GDP Output Shift`)**: Fits a Ridge-stabilized piecewise-linear Preston regression on each bootstrap sample, with the high-income diminishing-returns knot placed at the sample 75th percentile of `log_gdp_pcap` (`preston_knot = $27.5k`).
3. **Stage 2 — Robinson (1988) Partially Linear Within-Country R&D Frontier (`GenAI GDP + R&D/Health Spillover`)**:
   - **Endogenous R&D Shift**: From the 1995–2019 U.S. time series, each `+1.0` log-unit of GDP per capita is associated with `usa_rd_slope = +2.971 pp` of R&D share (`cov/var`). An incremental AI boost `g_ai` lifts U.S. R&D intensity by `usa_rd_slope * dlg_ai` (from `2.83%` up to `3.41%` at `+2.0%/yr`).
   - **De-Meaned Peer Frontier Kernel**: Across high-income R&D peers (top quartile of 2019 GDP/capita `gdp_q75` and top decile of R&D share `rd_q90`), we partial out all non-R&D covariates using Stage 1 $\hat{\beta}$ and subtract each country's initial 1995–1999 baseline residual (`country_fe`). This isolates **within-country longitudinal longevity gains** as peers expand R&D, eliminating cross-country cultural/dietary confounding.
   - **Silverman Bandwidths**: The bivariate Nadaraya-Watson kernel uses Silverman's rule of thumb $h_{\text{rd}} = \hat{\sigma}_{\text{within}} \cdot n^{-1/5}$ (`~0.086 pp`) and peer log-GDP standard deviation $h_{\text{lg}}$ (`~0.21` log-units).

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

# Collect the 4,667-row country-year panel to broadcast via closure to Spark executors
pdf_panel = (
    train_df.select("country_code", "year", "gdp_pcap", "life_expectancy", *feature_cols)
    .orderBy("country_code", "year")
    .toPandas()
)
medians = pdf_panel[feature_cols].median().to_dict()
for col_name in feature_cols:
    pdf_panel[col_name] = pdf_panel[col_name].fillna(medians[col_name])

# 1. Empirical U.S. parameters estimated directly from the historical 1995-2019 U.S. WDI series:
#    - g_base (1.83%/yr): U.S. 5-year rolling GDP-per-capita CAGR in 2019 (baseline non-AI growth rate)
#    - usa_rd_slope (2.971 pp per log-GDP unit): OLS slope cov(log_gdp_pcap, rd_pct_gdp) / var(log_gdp_pcap)
usa_hist = pdf_panel[pdf_panel["country_code"] == "USA"].sort_values("year")
usa_2019 = usa_hist[usa_hist["year"] == 2019].iloc[0].to_dict()
g_base = float(usa_2019["gdp_5yr_cagr_pct"])
usa_rd_slope = float(
    np.cov(usa_hist["log_gdp_pcap"], usa_hist["rd_pct_gdp"])[0, 1]
    / np.var(usa_hist["log_gdp_pcap"])
)

# 2. Sample quantile thresholds from the WDI panel (zero hardcoded dollar/R&D cutoffs):
#    - preston_knot: 75th percentile of log_gdp_pcap across the full panel ($27.5k PPP high-income transition)
#    - gdp_q75, rd_q90: 2019 top-quartile GDP/capita ($31.8k) and top-decile R&D share (1.90% of GDP)
#      selecting the 20 high-income innovation peer economies while excluding low-R&D petrostates
preston_knot = float(pdf_panel["log_gdp_pcap"].quantile(0.75))
p2019_all = pdf_panel[pdf_panel["year"] == 2019]
gdp_q75 = float(p2019_all["gdp_pcap"].quantile(0.75))
rd_q90 = float(p2019_all["rd_pct_gdp"].quantile(0.90))

HORIZON_YEARS = 10
N_BOOTSTRAP = 10_000
GDP_BOOST_STEPS = [round(0.1 * i, 1) for i in range(0, 21)]  # +0.0% to +2.0%/yr AI boost above g_base
panel_payload = pdf_panel.to_dict(orient="list")

boot_schema = StructType([
    StructField("boot_id", IntegerType(), False),
    StructField("scenario", StringType(), False),
    StructField("gdp_boost_pct", DoubleType(), False),
    StructField("projected_us_le", DoubleType(), False),
    StructField("gain_years", DoubleType(), False),
    StructField("gain_days", DoubleType(), False),
])

def simulate_country_cluster_bootstrap(iterator):
    panel = pd.DataFrame(panel_payload)
    countries = panel["country_code"].unique()
    n_countries = len(countries)
    country_blocks = {code: grp for code, grp in panel.groupby("country_code", sort=False)}

    def design_matrix(df_in: pd.DataFrame) -> np.ndarray:
        lg = df_in["log_gdp_pcap"].to_numpy(dtype=float)
        cagr = df_in["gdp_5yr_cagr_pct"].to_numpy(dtype=float)
        rd = df_in["rd_pct_gdp"].to_numpy(dtype=float)
        hlth = df_in["health_exp_pct_gdp"].to_numpy(dtype=float)
        net = df_in["internet_users_pct"].to_numpy(dtype=float)
        tert = df_in["tertiary_enroll_pct"].to_numpy(dtype=float)
        urb = df_in["urban_pop_pct"].to_numpy(dtype=float)
        yr = df_in["year"].to_numpy(dtype=float) - 1995.0
        return np.column_stack([
            np.ones_like(lg),
            lg,
            np.maximum(0.0, lg - preston_knot),  # Piecewise linear hinge above 75th percentile of log(GDP/capita)
            cagr,
            rd,
            hlth,
            net,
            tert,
            urb,
            yr,
        ])

    blocks_X = {code: design_matrix(grp) for code, grp in country_blocks.items()}
    blocks_y = {code: grp["life_expectancy"].to_numpy(dtype=float) for code, grp in country_blocks.items()}

    usa_actual_le = float(usa_2019["life_expectancy"])
    base_2019_X = design_matrix(pd.DataFrame([usa_2019]))
    base_rd = float(usa_2019["rd_pct_gdp"])
    base_lg = float(usa_2019["log_gdp_pcap"])

    # Balanced Non-AI 10-Year Baseline (g_ai = 0.0%):
    # U.S. GDP per capita compounds for 10 years at its baseline CAGR g_base (1.83%/yr -> $75.1k),
    # while expenditure shares (% of GDP) remain constant at their 2019 baseline shares.
    dlg_base = HORIZON_YEARS * np.log(1.0 + g_base / 100.0)
    usa_no_ai = dict(usa_2019)
    usa_no_ai["log_gdp_pcap"] = base_lg + dlg_base
    usa_no_ai["gdp_5yr_cagr_pct"] = g_base
    usa_no_ai["rd_pct_gdp"] = base_rd
    no_ai_X = design_matrix(pd.DataFrame([usa_no_ai]))

    for batch in iterator:
        rows = []
        for boot_id in batch["id"].to_numpy():
            # Block-resample whole country trajectories with replacement to preserve within-country serial correlation
            rng = np.random.default_rng(2026 + int(boot_id))
            sampled = rng.choice(countries, size=n_countries, replace=True)

            # Stage 1 (Parametric Preston Panel): Fit Ridge-stabilized panel regression on resampled country blocks
            X_b = np.vstack([blocks_X[c] for c in sampled])
            y_b = np.concatenate([blocks_y[c] for c in sampled])
            ridge = np.eye(X_b.shape[1]) * 1e-3
            ridge[0, 0] = 0.0
            beta = np.linalg.solve(X_b.T @ X_b + ridge, X_b.T @ y_b)

            pred_2019 = float((base_2019_X @ beta)[0])
            pred_no_ai = float((no_ai_X @ beta)[0])
            # Projected U.S. life expectancy after 10 years of Non-AI Baseline growth (79.23 yrs mean)
            le_no_ai = usa_actual_le + (pred_no_ai - pred_2019)

            # Stage 2 (Semiparametric Robinson Partially Linear + Within-Country FE Kernel):
            # Partial out non-R&D covariates using Stage 1 beta, then subtract each peer country's initial
            # 1995-1999 baseline residual (`country_fe`) so cultural/dietary differences (e.g., Japan vs. U.S.)
            # are not confounded with longitudinal R&D gains.
            peer_arrays = []
            for c in sampled:
                grp = country_blocks[c]
                rd_c = grp["rd_pct_gdp"].to_numpy(dtype=float)
                peer_mask = (grp["gdp_pcap"].to_numpy() >= gdp_q75) & (rd_c >= rd_q90)
                if not peer_mask.any():
                    continue

                X_c = blocks_X[c]
                y_c = blocks_y[c]
                fitted_non_rd = (X_c @ beta) - (rd_c * beta[4])
                rd_plus_resid = y_c - fitted_non_rd
                early_mask = grp["year"].to_numpy() <= 1999
                country_fe = (
                    float(np.mean(rd_plus_resid[early_mask]))
                    if early_mask.any()
                    else float(np.mean(rd_plus_resid))
                )
                within_rd_eff = rd_plus_resid - country_fe
                rd_within_dev = rd_c - float(np.mean(rd_c))

                peer_arrays.append(
                    np.column_stack([
                        grp["log_gdp_pcap"].to_numpy(dtype=float)[peer_mask],
                        rd_c[peer_mask],
                        rd_within_dev[peer_mask],
                        within_rd_eff[peer_mask],
                    ])
                )

            peer_data = np.vstack(peer_arrays)
            hi_lg = peer_data[:, 0]
            hi_rd = peer_data[:, 1]
            hi_rd_within = peer_data[:, 2]
            hi_eff = peer_data[:, 3]

            # Data-driven Nadaraya-Watson kernel bandwidths via Silverman's (1986) rule of thumb:
            # - h_rd: std(within-country R&D) * n^(-1/5) (~0.086 pp)
            # - h_lg: peer-group log(GDP/capita) standard deviation (~0.21 log units)
            n_p = len(peer_data)
            h_rd = float(np.std(hi_rd_within) * (n_p ** (-0.2)))
            h_lg = float(np.std(hi_lg))

            rd_eval = []
            for g_ai in GDP_BOOST_STEPS:
                dlg_tot = HORIZON_YEARS * np.log(1.0 + (g_base + g_ai) / 100.0)
                dlg_ai = dlg_tot - dlg_base
                r_target = base_rd + usa_rd_slope * dlg_ai
                lg_target = base_lg + dlg_ai
                w = np.exp(-0.5 * ((hi_rd - r_target) / h_rd) ** 2 - 0.5 * ((hi_lg - lg_target) / h_lg) ** 2)
                rd_eval.append(float(np.sum(w * hi_eff) / np.maximum(np.sum(w), 1e-12)))

            rd_eval = np.array(rd_eval)
            # Cumulative non-decreasing technological frontier shift above the g_ai = 0.0% baseline
            rd_regime_gain = np.maximum.accumulate(rd_eval - rd_eval[0])

            x_pure = no_ai_X.copy()
            for idx, g_ai in enumerate(GDP_BOOST_STEPS):
                lg_tot = base_lg + HORIZON_YEARS * np.log(1.0 + (g_base + g_ai) / 100.0)
                x_pure[0, 1] = lg_tot
                x_pure[0, 2] = max(0.0, lg_tot - preston_knot)
                x_pure[0, 3] = g_base + g_ai

                # Incremental longevity gain attributable to GenAI (above the Non-AI Baseline le_no_ai)
                gain_pure_yr = float((x_pure @ beta)[0]) - pred_no_ai
                rows.append((
                    int(boot_id),
                    "Pure GDP Output Shift",
                    float(g_ai),
                    le_no_ai + gain_pure_yr,
                    gain_pure_yr,
                    gain_pure_yr * 365.25,
                ))

                gain_spillover_yr = gain_pure_yr + float(rd_regime_gain[idx])
                rows.append((
                    int(boot_id),
                    "GenAI GDP + R&D/Health Spillover",
                    float(g_ai),
                    le_no_ai + gain_spillover_yr,
                    gain_spillover_yr,
                    gain_spillover_yr * 365.25,
                ))
        yield pd.DataFrame(
            rows,
            columns=["boot_id", "scenario", "gdp_boost_pct", "projected_us_le", "gain_years", "gain_days"],
        )

forecast_df = (
    spark.range(0, N_BOOTSTRAP, 1, numPartitions=64)
    .mapInPandas(simulate_country_cluster_bootstrap, schema=boot_schema)
    .groupBy("scenario", "gdp_boost_pct")
    .agg(
        F.avg("projected_us_le").alias("projected_us_le"),
        F.avg("gain_years").alias("mean_gain_years"),
        F.percentile_approx("gain_years", 0.10).alias("p10_gain_years"),
        F.percentile_approx("gain_years", 0.90).alias("p90_gain_years"),
        F.avg("gain_days").alias("mean_gain_days"),
        F.percentile_approx("gain_days", 0.10).alias("p10_gain_days"),
        F.percentile_approx("gain_days", 0.90).alias("p90_gain_days"),
    )
    .orderBy("scenario", "gdp_boost_pct")
)

forecast_pdf = forecast_df.toPandas()
forecast_df.filter(F.col("gdp_boost_pct").isin(0.0, 0.1, 0.5, 1.0, 1.5, 2.0)).select(
    "scenario",
    "gdp_boost_pct",
    F.round("projected_us_le", 2).alias("projected_us_le_yrs"),
    F.round("mean_gain_days", 1).alias("ai_gain_days"),
    F.round("p10_gain_days", 1).alias("p10_low_days"),
    F.round("p90_gain_days", 1).alias("p90_high_days"),
).show(truncate=False)

## 5. Visualize 10-Year Counterfactual Trajectories & Bootstrap Gains

Plot the two-panel forecast summary: **Panel A** places the 10-year U.S. Non-AI Baseline (`$62.6k → $75.1k`) and both GenAI counterfactuals alongside historical peer trajectories (`CHE`, `JPN`, `DEU`), while **Panel B** zooms into the AI-attributable days gained per `+0.1%/yr` productivity boost with 10th–90th percentile bootstrap bands.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
fig, (ax_left, ax_right) = plt.subplots(
    1, 2, figsize=(14.4, 5.7), dpi=150, gridspec_kw={"width_ratios": [1.12, 1.0]}
)

sp_sub = forecast_pdf[forecast_pdf["scenario"] == "GenAI GDP + R&D/Health Spillover"].sort_values("gdp_boost_pct")
pu_sub = forecast_pdf[forecast_pdf["scenario"] == "Pure GDP Output Shift"].sort_values("gdp_boost_pct")

boosts = sp_sub["gdp_boost_pct"].to_numpy()
sp_mean = sp_sub["mean_gain_days"].to_numpy()
sp_p10 = sp_sub["p10_gain_days"].to_numpy()
sp_p90 = sp_sub["p90_gain_days"].to_numpy()

pu_mean = pu_sub["mean_gain_days"].to_numpy()
pu_p10 = pu_sub["p10_gain_days"].to_numpy()
pu_p90 = pu_sub["p90_gain_days"].to_numpy()

# --- Panel A (LHS): Historical Trajectories + Non-AI Baseline + 10-Yr AI Counterfactuals in Phase Space ---
p2019 = pdf_panel[pdf_panel["year"] == 2019]
ax_left.scatter(
    p2019["gdp_pcap"] / 1000.0,
    p2019["life_expectancy"],
    color="#d1d5db",
    s=26,
    alpha=0.58,
    label="Global Economies (WDI Cross-Section)",
)

for code, col, ls, lbl in [
    ("CHE", "#6b7280", ":", "Switzerland Historical Path (\\$49k → \\$71k)"),
    ("JPN", "#0f766e", "-.", "Japan Historical Path (\\$34k → \\$42k)"),
    ("DEU", "#9ca3af", "--", "Germany Historical Path (\\$37k → \\$54k)"),
]:
    sub = pdf_panel[(pdf_panel["country_code"] == code) & (pdf_panel["year"] >= 1995) & (pdf_panel["year"] <= 2019)].sort_values("year")
    ax_left.plot(sub["gdp_pcap"] / 1000.0, sub["life_expectancy"], color=col, linestyle=ls, linewidth=1.9, label=lbl)

usa_sub = pdf_panel[(pdf_panel["country_code"] == "USA") & (pdf_panel["year"] >= 1995) & (pdf_panel["year"] <= 2019)].sort_values("year")
ax_left.plot(
    usa_sub["gdp_pcap"] / 1000.0,
    usa_sub["life_expectancy"],
    color="#111827",
    linewidth=2.6,
    marker="o",
    markersize=3.8,
    label="U.S. Historical Trajectory (\\$43k → \\$63k)",
)

x0 = float(usa_2019["gdp_pcap"]) / 1000.0
y0 = float(usa_2019["life_expectancy"])

# 1. Non-AI 10-Year Baseline trajectory (g_ai = 0.0%, economy grows at historical g_base = 1.83%/yr)
x_no_ai = x0 * ((1.0 + g_base / 100.0) ** HORIZON_YEARS)
y_no_ai = float(sp_sub["projected_us_le"].iloc[0])
ax_left.plot(
    [x0, x_no_ai],
    [y0, y_no_ai],
    color="#4b5563",
    linestyle=(0, (4, 2)),
    linewidth=2.2,
    label=f"U.S. {HORIZON_YEARS}-Yr Non-AI Baseline ({g_base:.2f}%/yr GDP Growth)",
)
ax_left.scatter([x_no_ai], [y_no_ai], color="#4b5563", marker="D", s=48, zorder=6, edgecolors="white", linewidth=0.8)

# 2. Counterfactual 10-Year endpoints as GenAI adds +0.0% to +2.0%/yr ON TOP of g_base (1.83%/yr)
gdp_fwd = x0 * ((1.0 + (g_base + boosts) / 100.0) ** HORIZON_YEARS)
le_genai_mean = sp_sub["projected_us_le"].to_numpy()
le_genai_low = y_no_ai + sp_sub["p10_gain_years"].to_numpy()
le_genai_high = y_no_ai + sp_sub["p90_gain_years"].to_numpy()
le_pure_mean = pu_sub["projected_us_le"].to_numpy()

ax_left.plot(
    gdp_fwd,
    le_genai_mean,
    color="#1a73e8",
    linewidth=2.6,
    marker="o",
    markersize=3.4,
    markevery=4,
    label="U.S. + GenAI GDP & R&D Spillover (+0.1% to +2.0%/yr)",
)
ax_left.fill_between(gdp_fwd, le_genai_low, le_genai_high, color="#1a73e8", alpha=0.17)
ax_left.plot(
    gdp_fwd,
    le_pure_mean,
    color="#e37400",
    linestyle="--",
    linewidth=2.2,
    label="U.S. + Pure AI GDP Growth Only (Fixed R&D Share)",
)

ax_left.scatter([x0], [y0], color="#dc2626", s=68, zorder=6, edgecolors="black")
ax_left.annotate(
    f"U.S. Initial\n(${x0:.1f}k, {y0:.1f}y)",
    xy=(x0, y0),
    xytext=(-6, 12),
    textcoords="offset points",
    ha="center",
    fontsize=8.2,
    fontweight="bold",
    color="#dc2626",
)
ax_left.annotate(
    f"No-AI Baseline\n(${x_no_ai:.1f}k, {y_no_ai:.2f}y)",
    xy=(x_no_ai, y_no_ai),
    xytext=(-36, -30),
    textcoords="offset points",
    fontsize=8.1,
    fontweight="bold",
    color="#374151",
    arrowprops=dict(arrowstyle="-", color="#9ca3af", lw=0.8),
)
ax_left.annotate(
    f"{le_genai_mean[-1]:.2f} yr",
    xy=(gdp_fwd[-1], le_genai_mean[-1]),
    xytext=(5, 2),
    textcoords="offset points",
    fontsize=8.8,
    fontweight="bold",
    color="#1a73e8",
)
ax_left.annotate(
    f"{le_pure_mean[-1]:.2f} yr",
    xy=(gdp_fwd[-1], le_pure_mean[-1]),
    xytext=(5, -5),
    textcoords="offset points",
    fontsize=8.6,
    fontweight="bold",
    color="#e37400",
)

ax_left.set_xlim(28, 99)
ax_left.set_ylim(74.5, 85.0)
ax_left.set_xlabel("Real GDP per Capita, PPP (Thousands of Constant International $)", fontsize=10)
ax_left.set_ylabel("Life Expectancy at Birth (Years)", fontsize=10)
ax_left.set_title("A. Country Trajectories & 10-Yr Counterfactuals in Phase Space", fontsize=11, fontweight="bold")
ax_left.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.94, fontsize=7.9)

# --- Panel B (RHS): Incremental Longevity Gain Attributable to GenAI (Above Non-AI Baseline) ---
ax_right.plot(
    boosts,
    sp_mean,
    color="#1a73e8",
    linewidth=2.7,
    marker="o",
    markersize=4.3,
    label=f"GenAI GDP + R&D / Healthcare Spillover ({N_BOOTSTRAP:,} Spark Replicates)",
)
ax_right.fill_between(
    boosts,
    sp_p10,
    sp_p90,
    color="#1a73e8",
    alpha=0.18,
    label="Bootstrap 10th–90th Percentile Band (WDI Peer Regimes)",
)

ax_right.plot(
    boosts,
    pu_mean,
    color="#e37400",
    linestyle="--",
    linewidth=2.3,
    marker="s",
    markersize=3.6,
    label="Pure AI GDP Output Shift Only (Fixed R&D & Health Share)",
)
ax_right.fill_between(boosts, pu_p10, pu_p90, color="#e37400", alpha=0.15)

for b_val, offset, halign in [(0.5, (-10, 14), "right"), (1.0, (-10, 14), "right"), (2.0, (7, 2), "left")]:
    idx = int(round(b_val * 10))
    y_val = sp_mean[idx]
    ax_right.annotate(
        f"+{y_val:.0f}d (+{y_val/365.25:.2f} yr)",
        xy=(b_val, y_val),
        xytext=offset,
        textcoords="offset points",
        ha=halign,
        fontsize=8.8,
        fontweight="bold",
        color="#1a73e8",
        arrowprops=dict(arrowstyle="-", color="#93c5fd", lw=0.9) if b_val < 2.0 else None,
    )

ax_right.annotate(
    f"+{pu_mean[-1]:.0f}d (+{pu_mean[-1]/365.25:.2f} yr)",
    xy=(2.0, pu_mean[-1]),
    xytext=(7, -2),
    textcoords="offset points",
    fontsize=8.8,
    fontweight="bold",
    color="#e37400",
)

ax_right.set_xlim(0.0, 2.45)
ax_right.set_ylim(-10, max(sp_p90) * 1.08)
ax_right.set_xticks(np.arange(0.0, 2.1, 0.2))
ax_right.set_xticklabels([f"+{x:.1f}%" for x in np.arange(0.0, 2.1, 0.2)])
ax_right.set_xlabel(f"Incremental GenAI Annual GDP-per-Capita Boost (+0.1%/yr above {g_base:.2f}%/yr Baseline)", fontsize=10)
ax_right.set_ylabel(f"AI-Attributable {HORIZON_YEARS}-Yr Longevity Gain Above Baseline (Days)", fontsize=10)
ax_right.set_title("B. Incremental Longevity Gain per +0.1%/yr GenAI Boost", fontsize=11, fontweight="bold")
ax_right.legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.94, fontsize=8.3)

fig.suptitle(
    f"Projected {HORIZON_YEARS}-Yr U.S. Longevity Gains from Generative AI Productivity (Serverless Spark WDI Panel Bootstrap, N={N_BOOTSTRAP:,})",
    fontsize=12.2,
    fontweight="bold",
    y=1.01,
)
fig.tight_layout()
plt.show()